# ARES 2023-68A — Per-Loan PD

Compute a **tenor-dependent probability of default (PD)** for each loan by mapping
its **S&P *issuer* rating** to cumulative default rates in `sp_default_rate.csv`,
then reading the rate at the loan's years-to-maturity. No plots.

**Key design choices (per brief):**
- **Key on issuer rating, not issue rating.** Issue ratings are notched for
  recovery; using them for PD would double-count what LGD already captures.
  Use `S&P Issuer Rating`; where missing, fall back to `S&P Reported Rating`
  (issue), **flagged**. If both are blank/NR → documented **conservative bucket**,
  flagged. Rating `D` → already defaulted → **PD = 1.0**, flagged.
- **Tenor-dependent.** `years_to_horizon = (Maturity Date − as_of)/365`,
  `as_of = 2026-05-31`. PD = cumulative default rate interpolated at that tenor.
- **Notch-exact mapping.** Keep B+/B/B- etc. distinct; only CCC+/CCC/CCC-/CC/C
  collapse to the table's single `CCC/C` bucket.

Output columns added to the derived CSV: `PD`, `pd_rating_used`,
`pd_rating_source`, `pd_bucket`, `pd_years_to_horizon`.
**Raw LLD (.xlsx) is never modified.**

In [1]:
import os, glob, re
import numpy as np
import pandas as pd

AS_OF = pd.Timestamp("2026-05-31")
UNRATED_BUCKET = "CCC/C"   # documented conservative bucket for truly-unrated loans

# locate inputs regardless of launch dir
def find(pattern, roots):
    for r in roots:
        hits = glob.glob(os.path.join(r, "**", pattern), recursive=True)
        if hits:
            return hits[0]
    return None

ROOTS = [".", "..", "../..", "../../..", "output", "../output"]
DERIVED_PATH = find("lld_ares_2023_derived.csv", ROOTS)
SP_PATH      = find("sp_default_rate.csv", ROOTS + ["../../data", "../../../data"])
print("derived LLD :", DERIVED_PATH)
print("default tbl :", SP_PATH)

derived LLD : ./output/lld_ares_2023_derived.csv
default tbl : ../../data/ARES_2023_68A/sp_default_rate.csv


In [2]:
# --- load inputs ---
lld = pd.read_csv(DERIVED_PATH)
sp  = pd.read_csv(SP_PATH)
print("LLD:", lld.shape, "| default table:", sp.shape)

# default-table rating keys and the year grid (cumulative % default at y1..y15)
YEAR_COLS = [c for c in sp.columns if re.fullmatch(r"y\d+", c)]
YEARS = np.array([int(c[1:]) for c in YEAR_COLS])                 # [1,2,...,15]
sp_idx = sp.set_index("rating")
TABLE_KEYS = set(sp_idx.index)
print("table keys:", sorted(TABLE_KEYS))

LLD: (415, 49) | default table: (17, 16)
table keys: ['A', 'A+', 'A-', 'AA', 'AA+', 'AA-', 'AAA', 'B', 'B+', 'B-', 'BB', 'BB+', 'BB-', 'BBB', 'BBB+', 'BBB-', 'CCC/C']


In [3]:
# --- rating string normalization -> a table key ---
def normalize(raw):
    """Return a sp_default_rate key, or 'D' (defaulted) / None (unrated)."""
    if pd.isna(raw):
        return None
    s = str(raw).strip().upper()
    if s in ("", "NR", "NAN", "N.A.", "NA", "WR"):
        return None
    s = s.split()[0]                       # drop outlook/watch tail e.g. 'B+ *-'
    s = re.sub(r"[^A-Z+\-/]", "", s)        # keep letters, +, -, /
    if s in ("D", "SD"):
        return "D"
    if s.startswith("CCC") or s in ("CC", "C", "CCC/C"):
        return "CCC/C"                     # collapse only the CCC/C region
    return s                               # AAA..B- pass through as-is

def blank(series):
    n = series.apply(normalize)
    return n.isna()


In [4]:
# --- pick rating per loan: issuer -> issue fallback -> conservative ---
def resolve(row):
    iss = normalize(row.get("S&P Issuer Rating"))
    rep = normalize(row.get("S&P Reported Rating"))
    if iss == "D" or rep == "D":
        return pd.Series({"pd_bucket": "D", "pd_rating_source": "defaulted",
                          "pd_rating_used": "D"})
    if iss is not None:
        return pd.Series({"pd_bucket": iss, "pd_rating_source": "issuer",
                          "pd_rating_used": iss})
    if rep is not None:
        return pd.Series({"pd_bucket": rep, "pd_rating_source": "issue_fallback",
                          "pd_rating_used": rep})
    return pd.Series({"pd_bucket": UNRATED_BUCKET,
                      "pd_rating_source": "unrated_conservative",
                      "pd_rating_used": UNRATED_BUCKET})

res = lld.apply(resolve, axis=1)
lld[["pd_bucket", "pd_rating_source", "pd_rating_used"]] = res
print(lld["pd_rating_source"].value_counts().to_string())

pd_rating_source
issuer                  389
issue_fallback           22
defaulted                 2
unrated_conservative      2


In [5]:
# --- tenor: years from as_of to maturity ---
mat = pd.to_datetime(lld["Maturity Date"], format="%m/%d/%Y", errors="coerce")
lld["pd_years_to_horizon"] = ((mat - AS_OF).dt.days / 365).clip(lower=0)
print("years-to-horizon: min %.2f max %.2f | unparsed maturities: %d"
      % (lld["pd_years_to_horizon"].min(), lld["pd_years_to_horizon"].max(), mat.isna().sum()))

years-to-horizon: min 0.03 max 6.92 | unparsed maturities: 0


/var/folders/vj/_0bszh_n0psbn0345y9vlrs40000gn/T/ipykernel_88272/3778393056.py:3: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  lld["pd_years_to_horizon"] = ((mat - AS_OF).dt.days / 365).clip(lower=0)


In [6]:
# --- interpolate cumulative default rate at each loan's tenor ---
# build xp=[0,1,..,15], fp=[0, y1,.., y15]/100 per bucket; np.interp caps at ends.
XP = np.concatenate([[0.0], YEARS.astype(float)])

def pd_for(bucket, years):
    if bucket == "D":
        return 1.0                                    # already defaulted
    row = sp_idx.loc[bucket, YEAR_COLS].to_numpy(dtype=float) / 100.0
    fp = np.concatenate([[0.0], row])
    return float(np.interp(years, XP, fp))

lld["PD"] = [pd_for(b, y) for b, y in zip(lld["pd_bucket"], lld["pd_years_to_horizon"])]
print(lld[["Issuer", "pd_rating_used", "pd_rating_source",
           "pd_years_to_horizon", "PD"]].head(10).to_string(index=False))
print("\nPD range:", round(lld["PD"].min(), 4), "->", round(lld["PD"].max(), 4))

                        Issuer pd_rating_used pd_rating_source  pd_years_to_horizon       PD
Freeport LNG Investments, LLLP             B-           issuer             6.706849 0.248784
                    Ensono, LP              B           issuer             1.989041 0.063198
        Tempo Acquisition, LLC             B+           issuer             2.254795 0.057548
     Telenet Financing USD LLC            BB-           issuer             1.912329 0.025487
       SCIH Salt Holdings Inc.              B           issuer             2.673973 0.085976
              Proofpoint, Inc.             B-           issuer             2.254795 0.124077
   Epicor Software Corporation             B-           issuer             5.000000 0.220300
                RealPage, Inc.             B-           issuer             1.901370 0.106043
                 Xplor T1, LLC             B-           issuer             6.509589 0.246081
         Citadel Securities LP           BBB-           issuer        

/var/folders/vj/_0bszh_n0psbn0345y9vlrs40000gn/T/ipykernel_88272/2470122173.py:12: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  lld["PD"] = [pd_for(b, y) for b, y in zip(lld["pd_bucket"], lld["pd_years_to_horizon"])]


In [7]:
# --- write PD (and supporting cols) back to the derived CSV; raw .xlsx untouched ---
lld.to_csv(DERIVED_PATH, index=False)
print("wrote", os.path.abspath(DERIVED_PATH))
print("rows:", len(lld), "| new cols: PD, pd_rating_used, pd_rating_source, "
      "pd_bucket, pd_years_to_horizon")
# quick audit
print("\nby source:")
print(lld.groupby("pd_rating_source")["PD"].agg(["count", "mean"]).round(4).to_string())

wrote /Users/amine/Documents/Columbia/Classes/ENGIE 4700 - Summer Project/correlation_and_tail_risk_in_clo_tranches/project/notebooks/output/lld_ares_2023_derived.csv
rows: 415 | new cols: PD, pd_rating_used, pd_rating_source, pd_bucket, pd_years_to_horizon

by source:
                      count    mean
pd_rating_source                   
defaulted                 2  1.0000
issue_fallback           22  0.1252
issuer                  389  0.1369
unrated_conservative      2  0.4382
